# Adult: Pipeline of Dataset Group Fairness and Classifier Utility & Group Fairness Results

Author: Ilse Harmers \
Last modified: February 26, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix

In [ ]:
# Initializing the synthesizer for preprocessing the datasets with the TableTransformer function. 
synth = Synthesizer.create('pategan', epsilon = 10.0, delta = 1e-05, plot_losses = True)

In [ ]:
# Importing the real training dataset.
adult_train = pd.read_csv("./train-test-datasets/adult/adult_train.csv")

# One-hot encoding the categorical columns in the training dataset.  
cat_columns = adult_train.select_dtypes(include=['object']).columns.to_list()
train_cat_encoded = utils.one_hot_encode(adult_train[cat_columns], order = [[" <=50K", " >50K"], [" Female", " Male"]])
#print(train_cat_encoded.shape)

# Transforming the continuous columns with the TableTransformer function.
tt = tf.TableTransformer([  
    tf.MinMaxTransformer(lower = adult_train["age"].min(), upper = adult_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_train["capital-gain"].max() + 1),
                             negative = False) # capital-gain; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_train["capital-loss"].max() + 1),
                             negative = False) # capital-loss; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = adult_train["hours-per-week"].min(), 
                         upper = adult_train["hours-per-week"].max(), negative = False), # hours-per-week
])

tf_cont_cols = np.array(synth._get_train_data(
                    adult_train[["age", "capital-gain", "capital-loss", "hours-per-week"]], transformer = tt, 
                    preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                    ordinal_columns=None, continuous_columns=None,))

num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "capital-gain", "capital-loss", "hours-per-week"])

# Concatenating the categorical and numerical columns back together.
train_encoded = pd.concat([train_cat_encoded.reset_index(drop = True), num_encoded.reset_index(drop = True), 
                           adult_train[["education-num"]].reset_index(drop = True)], axis = 1)

#train_encoded

In [ ]:
# Importing the real test set.
adult_test = pd.read_csv("./train-test-datasets/adult/adult_test.csv")

# One-hot encoding the categorical columns in the test dataset.
test_cat_encoded = utils.one_hot_encode(adult_test[cat_columns], order = [[" <=50K", " >50K"], [" Female", " Male"]])
#print(test_cat_encoded.shape)

# Checking for missing columns between the real training and test datasets. 
missing_cols = list(set(list(train_cat_encoded.columns)) - set(list(test_cat_encoded.columns)))
print("Missing column(s):", missing_cols)
# If there is a missing column, most likely due to a label from the training dataset not being present in the test set, then 
# we reinsert a 'zero' column into the test set at the right place to account for this missing label.
if missing_cols:
    for c in missing_cols:
        df_col = pd.DataFrame({c: adult_test["age"]*0})
        test_cat_encoded.insert(0, c, value = df_col)
test_cat_encoded = test_cat_encoded[train_cat_encoded.columns]
#print(test_cat_encoded.shape)

# Transforming the continuous columns with the TableTransformer function.
tt = tf.TableTransformer([  
    tf.MinMaxTransformer(lower = adult_test["age"].min(), upper = adult_test["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_test["capital-gain"].max() + 1),
                             negative = False) # capital-gain; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(adult_test["capital-loss"].max() + 1),
                             negative = False) # capital-loss; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = adult_test["hours-per-week"].min(), upper = adult_test["hours-per-week"].max(),
                         negative = False), # hours-per-week; scaling to range (0, 1)
])

tf_cont_cols = np.array(synth._get_train_data(
                    adult_test[["age", "capital-gain", "capital-loss", "hours-per-week"]], transformer = tt, 
                    preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                    ordinal_columns=None, continuous_columns=None,))

num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "capital-gain", "capital-loss", "hours-per-week"])

# Concatenating the categorical and numerical columns back together.
test_encoded = pd.concat([test_cat_encoded.reset_index(drop = True), num_encoded.reset_index(drop = True), 
                            adult_test[["education-num"]].reset_index(drop = True)], axis = 1)

#test_encoded

In [ ]:
# Setting up pipeline variables.
epsi = [1, 2, 5, 8]   # or [0], if the GAN in question is not differentially private, or ["real"], if calling the real dataset
runs = range(1, 6)  
samples = range(1, 4)

# Initializing file path to the synthetic datasets.
#file_path = "./train-test-datasets/adult/"
#file_path = "./synthetic-datasets_DP-GAN/adult/"
#file_path = "./synthetic-datasets_DPCTGAN/adult/"
#file_path = "./synthetic-datasets_FairDP-GAN(dem)/adult/"
#file_path = "./synthetic-datasets_FairDP-GAN(dis)/adult/"
#file_path = "./synthetic-datasets_FairDP-GAN(dem-dis)/adult/"
#file_path = "./synthetic-datasets_TabFair/adult/"

# Initializing our classifiers.
rf = RandomForestClassifier()
mlp = MLPClassifier(max_iter = 500)
lda = LinearDiscriminantAnalysis()

# Initializing dictionary for confusion matrix title.
clfs = {rf: "rf", mlp: "mlp", lda: "lda"}
clf_names = {rf: "Random Forest Classifier", mlp: "Multi-Layer Perceptron Classifier", lda: "LDA Classifier"}

In [ ]:
# This cell contains the pipeline with which the dataset group fairness, classifier group fairness and classifier utility results are computed.
# These results will be stored as csv files in the corresponding folders (see 'filepath'); these folders must exist for this pipeline to work properly.
for e in epsi:
    for r in runs:
        print(f"Run {r}")
        results = {"dem-parity": [], "dis-impact": []}
        for s in samples:
        
            # Importing synthetic dataset.
            if e == 0:
                sample = pd.read_csv(file_path + f"run{r}/sample{s}.csv")
            elif e == "real":   # or copying the real dataset
                sample = adult_train.copy()
            else:
                sample = pd.read_csv(file_path + f"epsi_{e}/run{r}/sample{s}.csv")

            # Occasionally, it happens that "Male" or "Female" is recorded in a synthetic dataset instead of " Male" or " Female". 
            # These lines rectify those mistakes. 
            if sample["sex"].loc[sample["sex"] == "Male"].shape[0] > 0:
                #print(sample["sex"].loc[sample["sex"] == "Male"].shape[0])
                sample["sex"].loc[sample["sex"] == "Male"] = " Male"
                #print(sample["sex"].loc[sample["sex"] == "Male"].shape[0])
            if sample["sex"].loc[sample["sex"] == "Female"].shape[0] > 0:
                #print(sample["sex"].loc[sample["sex"] == "Female"].shape[0])
                sample["sex"].loc[sample["sex"] == "Female"] = " Female"
                #print(sample["sex"].loc[sample["sex"] == "Female"].shape[0])

            # Encoding the sensitive and target attributes for the fairness analysis.
            sample_cols_encoded = utils.one_hot_encode(sample[["sex", "income"]], order = [[" <=50K", " >50K"], [" Female", " Male"]])
        
            dem_sample = utils.demographic_parity(df = sample_cols_encoded, s = "sex", y = "income")
            results["dem-parity"].append(dem_sample)

            dis_sample = utils.disparate_impact(df = sample_cols_encoded, s = "sex", y = "income")
            results["dis-impact"].append(dis_sample)

            # One-hot encoding the categorical columns in the synthetic dataset. 
            cat_columns = sample.select_dtypes(include=['object']).columns.to_list()
            sample_cat_encoded = utils.one_hot_encode(sample[cat_columns], order = [[" <=50K", " >50K"], [" Female", " Male"]])
        
            # Checking for missing columns between the train and synthetic sets. 
            missing_cols = list(set(list(train_cat_encoded.columns)) - set(list(sample_cat_encoded.columns)))
            # If there is a missing column, most likely due to a label from the train set not being present in the synthetic set, 
            # then we reinsert a 'zero' column into the synthetic set at the right place to account for this missing label.
            if missing_cols:
                for c in missing_cols:
                    df_col = pd.DataFrame({c: sample["age"]*0})
                    sample_cat_encoded.insert(0, c, value = df_col)

            # Transforming the continuous columns with the TableTransformer function.
            tt = tf.TableTransformer([  
                tf.MinMaxTransformer(lower = sample["age"].min(), upper = sample["age"].max(), 
                                     negative = False), # age; scaling to range (0, 1)
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = 0, upper = np.log(sample["capital-gain"].max() + 1),
                                         negative = False) # capital-gain; scaling to range (0, 1)
                ]),
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = 0, upper = np.log(sample["capital-loss"].max() + 1),
                                         negative = False) # capital-loss; scaling to range (0, 1)
                ]),
                tf.MinMaxTransformer(lower = sample["hours-per-week"].min(), upper = sample["hours-per-week"].max(),
                                     negative = False), # hours-per-week; scaling to range (0, 1)
            ])

            tf_cont_cols = np.array(synth._get_train_data(
                                sample[["age", "capital-gain", "capital-loss", "hours-per-week"]], transformer = tt, 
                                preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                                ordinal_columns=None, continuous_columns=None,))

            num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "capital-gain", "capital-loss", "hours-per-week"])

            # Concatenating the categorical and numerical columns back together.
            sample_encoded = pd.concat([sample_cat_encoded.reset_index(drop = True), num_encoded.reset_index(drop = True), 
                                        sample[["education-num"]].reset_index(drop = True)], axis = 1)
            sample_encoded = sample_encoded[train_encoded.columns]

            # Saving the encoded datasets (real or synthetic) for future objectives.
            if e == 0:
                sample_encoded.to_csv(file_path + f"run{r}/sample{s}_encoded.csv", index = False)
            elif e == "real":
                sample_encoded = train_encoded.copy()
                sample_encoded.to_csv(file_path + "train_encoded.csv", index = False)
                test_encoded.to_csv(file_path + "test_encoded.csv", index = False)
            else:
                sample_encoded.to_csv(file_path + f"epsi_{e}/run{r}/sample{s}_encoded.csv", index = False)
            
            for c in clfs:
                np.random.seed(42)   # setting random seed for reproducibility
                # Training an ML classifier and generating its predictions.
                model_synth = c.fit(sample_encoded.drop(columns = ["income"]), sample_encoded["income"])
                preds_synth = model_synth.predict(test_encoded.drop(columns = ["income"]))

                # Accuracy score.
                acc = accuracy_score(test_encoded["income"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-acc"] = [acc]
                else:
                    results[f"{clfs[c]}-acc"].append(acc)
                
                # Precision score.
                prec = precision_score(test_encoded["income"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-prec"] = [prec]
                else:
                    results[f"{clfs[c]}-prec"].append(prec)
                
                # Recall score.
                recall = recall_score(test_encoded["income"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-recall"] = [recall]
                else:
                    results[f"{clfs[c]}-recall"].append(recall)

                # AUROC score.
                auroc = roc_auc_score(test_encoded["income"], preds_synth)
                if s == 1:
                    results[f"{clfs[c]}-auroc"] = [auroc]
                else:
                    results[f"{clfs[c]}-auroc"].append(auroc)

                # Confusion matrix.
                conf_mat = confusion_matrix(test_encoded["income"], preds_synth)
                # Plotting the confusion matrix of the ML classifier.
                fig = sns.heatmap(conf_mat, annot=True, cmap = "mako", fmt='g')
                fig.set_title(f"Confusion Matrix of {clf_names[c]} \n ($\epsilon = {e}$; run{r}/sample{s})")
                fig.set_xlabel("Predicted Label")
                fig.set_ylabel("True Label")
                fig.set_xticks(ticks = [0.5, 1.5], labels = ["<=50K", ">50K"])   # setting ticks on x-axis to match target labels
                fig.set_yticks(ticks = [0.5, 1.5], labels = ["<=50K", ">50K"])   # setting ticks on y-axis to match target labels
                if e == 0:
                    plt.savefig(file_path + f"run{r}/confs/sample{s}_conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                elif e == "real":
                    plt.savefig(file_path + f"confs/conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                else:
                    plt.savefig(file_path + f"epsi_{e}/run{r}/confs/sample{s}_conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                plt.close()
                
                # Creating a DataFrame with the encoded 'sex' column and the classifier's predictions.
                preds_synth = pd.Series(preds_synth, name = "predicted_income")
                test_cols = pd.concat([test_encoded[["sex", "income"]].reset_index(drop = True), preds_synth.reset_index(drop = True)], axis = 1)

                # Demographic parity.
                dem_clf = utils.demographic_parity(df = test_cols, s = "sex", y = "predicted_income")
                if s == 1:
                    results[f"{clfs[c]}-dem"] = [dem_clf]
                else:
                    results[f"{clfs[c]}-dem"].append(dem_clf)

                # Disparate impact.
                try:
                    dis_clf = utils.disparate_impact(df = test_cols, s = "sex", y = "predicted_income")
                except ZeroDivisionError:
                    dis_clf = "np.nan"
                if s == 1:
                    results[f"{clfs[c]}-dis"] = [dis_clf]
                else:
                    results[f"{clfs[c]}-dis"].append(dis_clf)

                # Equal opportunity.
                eqopp_clf = utils.equal_opportunity(df = test_cols, s = "sex", y_true = "income", y_pred = "predicted_income")
                if s == 1:
                    results[f"{clfs[c]}-eqopp"] = [eqopp_clf]
                else:
                    results[f"{clfs[c]}-eqopp"].append(eqopp_clf)

        # Saving the 'results' dictionary as a csv file in the appropriate folder.
        df_results = pd.DataFrame(results).rename(index = {0: "sample1", 1: "sample2", 2: "sample3"})
        if e == 0:
            df_results.to_csv(file_path + f"run{r}/results_run{r}.csv")
        elif e == "real":
            df_results.to_csv(file_path + "results_real.csv")
        else:
            df_results.to_csv(file_path + f"epsi_{e}/run{r}/results_epsi-{e}_run{r}.csv")